# Étape 5 — Évaluation automatique (précision / rappel)

Compare le verdict produit par le pipeline (`src/detection.generer_verdict_dossier`)
au `ground_truth.json`, sur l'échantillon de test stratifié. Tolère un lot
partiellement traité (dossiers pas encore passés par l'étape 1).

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

NOM_DATASET = "dentaire"  # "maladie" ou "dentaire" -- un seul à la fois

import json
from pathlib import Path
import pandas as pd
from src.detection import generer_verdict_dossier

DATA_DIR = Path(f"../data/Dataset_{NOM_DATASET}")
OCR_RESULTS_DIR = Path(f"../outputs/ocr_results/{NOM_DATASET}")
ECHANTILLON_PATH = Path(f"../outputs/echantillon_test_{NOM_DATASET}.txt")

assert ECHANTILLON_PATH.exists(), f"{ECHANTILLON_PATH} introuvable -- lance d'abord 00_exploration_dataset.ipynb"


def charger_ocr(chemin: Path) -> dict:
    with open(chemin, "r", encoding="utf-8") as f:
        return json.load(f)

## Évaluation sur l'échantillon de test

In [2]:
with open(DATA_DIR / "ground_truth.json", "r", encoding="utf-8") as f:
    ground_truth = {d["fichier"]: d for d in json.load(f)}

noms_echantillon = ECHANTILLON_PATH.read_text(encoding="utf-8").splitlines()

resultats = []
verdicts_par_fichier = {}  # conservé pour les diagnostics détaillés plus bas
non_traites = []

for nom_fichier in noms_echantillon:
    chemin_json = OCR_RESULTS_DIR / f"{Path(nom_fichier).stem}_ocr.json"
    if not chemin_json.exists():
        non_traites.append(nom_fichier)
        continue

    dossier_ocr = charger_ocr(chemin_json)
    verdict = generer_verdict_dossier(dossier_ocr["pages"])
    verdicts_par_fichier[nom_fichier] = verdict
    verite = ground_truth[nom_fichier]

    predit_incoherent = not verdict["coherent"]
    reel_incoherent = not verite["coherent"]

    if predit_incoherent and reel_incoherent:
        categorie = "vrai_positif"
    elif predit_incoherent and not reel_incoherent:
        categorie = "faux_positif"
    elif not predit_incoherent and reel_incoherent:
        categorie = "faux_negatif"
    else:
        categorie = "vrai_negatif"

    resultats.append({
        "fichier": nom_fichier,
        "type_erreur_reel": verite["type_erreur"],
        "coherent_reel": verite["coherent"],
        "coherent_predit": verdict["coherent"],
        "categorie": categorie,
        "nb_anomalies_champs": len(verdict["anomalies_champs"]),
        "nb_pages_dupliquees": len(verdict["pages_dupliquees"]),
    })

df_resultats = pd.DataFrame(resultats)

print(f"{len(df_resultats)}/{len(noms_echantillon)} dossiers évalués "
      f"({len(non_traites)} pas encore traités par l'étape 1)")
if non_traites:
    print(f"  Non traités : {non_traites[:5]}{'...' if len(non_traites) > 5 else ''}")

45/45 dossiers évalués (0 pas encore traités par l'étape 1)


## Métriques globales

In [3]:
if len(df_resultats) == 0:
    print("Aucun dossier évalué pour l'instant -- relance cette cellule une fois l'étape 1 avancée.")
else:
    vp = (df_resultats["categorie"] == "vrai_positif").sum()
    fp = (df_resultats["categorie"] == "faux_positif").sum()
    fn = (df_resultats["categorie"] == "faux_negatif").sum()
    vn = (df_resultats["categorie"] == "vrai_negatif").sum()

    precision = vp / (vp + fp) if (vp + fp) > 0 else float("nan")
    rappel = vp / (vp + fn) if (vp + fn) > 0 else float("nan")
    f1 = 2 * precision * rappel / (precision + rappel) if (precision + rappel) > 0 else float("nan")
    exactitude = (vp + vn) / len(df_resultats)

    print("Matrice de confusion (positif = incohérent) :")
    print(f"  Vrais positifs  : {vp}")
    print(f"  Faux positifs   : {fp}  (dossiers cohérents signalés à tort)")
    print(f"  Faux négatifs   : {fn}  (incohérences ratées -- le cas le plus grave)")
    print(f"  Vrais négatifs  : {vn}")
    print()
    print(f"Précision : {precision:.2%}")
    print(f"Rappel    : {rappel:.2%}")
    print(f"F1-score  : {f1:.2%}")
    print(f"Exactitude globale : {exactitude:.2%}")

Matrice de confusion (positif = incohérent) :
  Vrais positifs  : 22
  Faux positifs   : 1  (dossiers cohérents signalés à tort)
  Faux négatifs   : 3  (incohérences ratées -- le cas le plus grave)
  Vrais négatifs  : 19

Précision : 95.65%
Rappel    : 88.00%
F1-score  : 91.67%
Exactitude globale : 91.11%


## Rappel par type d'erreur

In [4]:
if len(df_resultats) > 0:
    incoherents = df_resultats[df_resultats["coherent_reel"] == False]
    for type_erreur, groupe in incoherents.groupby("type_erreur_reel"):
        nb_total = len(groupe)
        nb_detecte = (groupe["categorie"] == "vrai_positif").sum()
        print(f"  {type_erreur:30s} : {nb_detecte}/{nb_total} détectés ({nb_detecte/nb_total:.0%})")

  date_naissance_mismatch        : 5/5 détectés (100%)
  document_autre_patient         : 3/5 détectés (60%)
  document_duplique              : 5/5 détectés (100%)
  nom_prenom_mismatch            : 5/5 détectés (100%)
  numero_dossier_mismatch        : 4/5 détectés (80%)


## Détail des erreurs (faux positifs et faux négatifs)

In [5]:
if len(df_resultats) > 0:
    fautifs = df_resultats[df_resultats["categorie"].isin(["faux_positif", "faux_negatif"])]
    if len(fautifs) == 0:
        print("Aucun faux positif ni faux négatif sur les dossiers évalués jusqu'ici.")
    else:
        print(fautifs[["fichier", "type_erreur_reel", "coherent_reel", "coherent_predit", "categorie"]]
              .to_string(index=False))

           fichier        type_erreur_reel  coherent_reel  coherent_predit    categorie
Dossier_000063.pdf numero_dossier_mismatch          False             True faux_negatif
Dossier_000216.pdf                     NaN           True            False faux_positif
Dossier_000256.pdf  document_autre_patient          False             True faux_negatif
Dossier_000554.pdf  document_autre_patient          False             True faux_negatif


## Diagnostic détaillé des faux négatifs

Utilise directement `verdict["details_champs"]` (déjà calculé lors de
l'évaluation, plus besoin de rappeler `comparer_dossier` séparément).

In [6]:
faux_negatifs = df_resultats[df_resultats["categorie"] == "faux_negatif"]["fichier"].tolist()

for nom_fichier in faux_negatifs:
    verdict = verdicts_par_fichier[nom_fichier]
    type_reel = ground_truth[nom_fichier]["type_erreur"]

    print("===", nom_fichier, "(type réel :", type_reel, ") ===")
    if not verdict["details_champs"]:
        print("  (aucun champ comparable trouvé sur 2+ pages -- rien à comparer)")
    for champ, detail in verdict["details_champs"].items():
        print("  ", champ, ": référence page", detail["page_reference"], repr(detail["valeur_reference"]))
        for c in detail["comparaisons"]:
            statut = "OK" if c["coherent"] else "ANOMALIE"
            print("     page", c["page"], repr(c["valeur"]), "score=", round(c["score_similarite"], 2), f"[{statut}]")
    print()

=== Dossier_000063.pdf (type réel : numero_dossier_mismatch ) ===
   nom_prenom : référence page 2 'EL GuEDDARI Meriem'
     page 3 'EL GUEDDARI Meriem' score= 1.0 [OK]
     page 4 'EL GUEDDARI Meriem' score= 1.0 [OK]

=== Dossier_000256.pdf (type réel : document_autre_patient ) ===
   numero_dossier : référence page 1 'DOS202600256'
     page 3 'DOs202600256' score= 1.0 [OK]
   nom_prenom : référence page 3 'FAssI FIHRI Rania'
     page 4 'FASSI FIHRI Rania' score= 1.0 [OK]
   date_naissance : référence page 1 '02/d8/1954'
     page 3 'p2/08/1954' score= 1.0 [OK]

=== Dossier_000554.pdf (type réel : document_autre_patient ) ===
   numero_dossier : référence page 1 'DOS202600554'
     page 3 'DOs202600554' score= 1.0 [OK]
   nom_prenom : référence page 2 'FASSI FIHRI Rania'
     page 3 'FAssI FIHRI Rania' score= 1.0 [OK]
   date_naissance : référence page 1 '2B/11/1991'
     page 3 '23/11/1991' score= 1.0 [OK]



## Diagnostic détaillé des faux positifs

In [7]:
faux_positifs = df_resultats[df_resultats["categorie"] == "faux_positif"]["fichier"].tolist()

for nom_fichier in faux_positifs:
    verdict = verdicts_par_fichier[nom_fichier]

    print("===", nom_fichier, "===")
    for a in verdict["anomalies_champs"]:
        print(
            "  [CHAMP]", a["champ"], ": page", a["page_reference"], f"({a['valeur_reference']!r})",
            "vs page", a["page_suspecte"], f"({a['valeur_suspecte']!r})",
            f"score={a['score_similarite']:.2f}",
        )
    for d in verdict["pages_dupliquees"]:
        print(
            "  [DOUBLON] page", d["page_a"], "<-> page", d["page_b"],
            f"similarité={d['similarite_texte']:.2f}",
        )
    print()

=== Dossier_000216.pdf ===
  [CHAMP] date_naissance : page 1 ('0B/1/1947') vs page 2 ('o3/l1/1947') score=0.80



## À faire ensuite

Pour ajuster un seuil ou une règle d'extraction suite à un diagnostic,
modifie directement le fichier `src/` concerné -- `field_extraction.py`,
`comparison.py` ou `detection.py` -- puis relance ce notebook.